In [1]:
!pip install --quiet "agno==2.6.21" "google-genai==2.10.0" "pytz" "gradio"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00


In [2]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("✅ Ключ завантажено")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google Gemini API key: ")

✅ Ключ завантажено


In [3]:
from agno.agent import Agent
from agno.models.google import Gemini
from agno.tools import tool
import gradio as gr

# 1. База даних
EXERCISES_DB = {
    "груди": {"вдома": [{"назва": "Віджимання від підлоги", "техніка": "Тіло пряме, опускайтесь до торкання", "підходи": "3-4", "повторення": "10-15"}]},
    "спина": {"вдома": [{"назва": "Підтягування", "техніка": "Хват ширше плечей", "підходи": "3", "повторення": "максимум"}]},
    "ноги":  {"вдома": [{"назва": "Присідання", "техніка": "Спина пряма, до паралелі", "підходи": "4", "повторення": "15-20"}]}
}

# 2. Інструменти
@tool
def exercise_lookup(query: str) -> str:
    """Шукає вправи за групою м'язів. Використовуй для підбору тренувань."""
    query_lower = query.lower()
    muscle_group = next((g for g in EXERCISES_DB if g in query_lower), None)

    if not muscle_group:
        return f"Групу м'язів не розпізнано. Доступні: {', '.join(EXERCISES_DB.keys())}."

    exercises = EXERCISES_DB[muscle_group].get("вдома", [])
    result = [f"🏋️ Вправи на {muscle_group}:"]
    for ex in exercises:
        result.append(f"\n🔹 {ex['назва']}\nТехніка: {ex['техніка']}\nПідходи: {ex['підходи']} × {ex['повторення']}")
    return "\n".join(result)

@tool
def calculate_target_heart_rate(age: int, resting_hr: int = 60) -> str:
    """Розраховує цільову зону пульсу для жироспалювання за формулою Карвонена."""
    max_hr = 220 - age
    reserve_hr = max_hr - resting_hr
    zone_min = resting_hr + (reserve_hr * 0.6)
    zone_max = resting_hr + (reserve_hr * 0.7)
    return f"❤️ Цільова зона пульсу для жироспалювання: {int(zone_min)} - {int(zone_max)} уд/хв."

# 3. Ініціалізація Агента
fitcoach_agent = Agent(
    name="FitCoach",
    model=Gemini(id="gemini-3.5-flash"), # Використовуємо найсвіжішу модель з твого списку
    tools=[exercise_lookup, calculate_target_heart_rate],
    instructions=[
        "Ти — персональний фітнес-асистент FitCoach.",
        "Завжди використовуй інструменти для підбору вправ або розрахунку пульсу.",
        "Не вигадуй власні вправи поза базою.",
        "Не давай медичних порад.",
        "Відповідай українською, чітко і привітно."
    ],
    markdown=True
)

# 4. Функція для інтерфейсу Gradio
def chat_with_bot(message, history):
    # Метод run повертає об'єкт RunResponse, беремо з нього контент
    response = fitcoach_agent.run(message)
    return response.content

# 5. Запуск Web-інтерфейсу
demo = gr.ChatInterface(
    fn=chat_with_bot,
    title="💪 FitCoach AI",
    description="Твій персональний фітнес-асистент. Запитай про вправи або розрахунок пульсу!",
    examples=["Які вправи на груди вдома?", "Мені 26 років, пульс у спокої 65. Який пульс тримати на пробіжці в Таллінні?"]
)

demo.launch(share=True) # share=True згенерує публічну лінку для твоєї презентації!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3281217f784fb431a7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
